# A mesterséges intelligencia tárgy eredményének előrejelzése

**Felügyelt gépi tanulás beadandó**

Azt vizsgálom, hogy a korábbi tanulmányi eredmények alapján mennyire lehet előre
jelezni a **Mesterséges intelligencia alapjai** tárgy féléves eredményét.
A két kiválasztott tárgykód minden megfigyelt felvételét vizsgálom, képzési szinttől
függetlenül. A kimenet az öt jegy valamelyike vagy az aláírás megtagadása.

## Előkészített adatok betöltése

A CSV egy eredménybejegyzés-alapú történeti tábla, nem közvetlen tanítómátrix.
Ugyanahhoz a hallgatóhoz sok tantárgy és több félév tartozhat. Azonosítókat
csak összekapcsolásra használok; nem adom őket a modelleknek.
A bemeneti fájl lenyomatát és a könyvtárverziókat is feljegyzem.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, os, platform, re, time, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, classification_report,
    ConfusionMatrixDisplay, make_scorer,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.inspection import permutation_importance

candidates = [Path.cwd(), Path.cwd() / "supervised_ml"]
PROJECT_DIR = next((p for p in candidates if (p / "data/eredmenyek.csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Nyisd meg a notebookot a supervised_ml vagy a repository mappából.")
CSV_PATH = PROJECT_DIR / "data/eredmenyek.csv"
RESULT_DIR = PROJECT_DIR / "data/notebook_results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_JOBS = min(4, os.cpu_count() or 1)
MI_CODES = {"VEMISA3242M", "VEMISAB254MV"}
CLASS_ORDER = ["Megtagadva", "Elégtelen (1)", "Elégséges (2)", "Közepes (3)", "Jó (4)", "Jeles (5)"]
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_colwidth", None)

records = pd.read_csv(CSV_PATH, dtype={"hallgato_id": "string", "kepzes_kod": "string"})
required = {"kollegium_elorejelzeskor"}
if not required <= set(records):
    raise ValueError("Előbb futtasd: python prepare_data.py --add-semester-context-only")
DATA_SHA256 = hashlib.file_digest(CSV_PATH.open("rb"), "sha256").hexdigest()
versions = {"Python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "scikit-learn": sklearn.__version__}
display(pd.DataFrame([{"Bejegyzések": len(records), "Hallgatók": records.hallgato_id.nunique(),
                       "Oszlopok": len(records.columns), "CSV méret (MB)": CSV_PATH.stat().st_size / 1e6}]).round(2))
display(pd.Series(versions, name="Verzió"))

## Tanítóminták előállítása a bejegyzésekből

Egy tanítópélda egy **hallgató–képzés–MI-félév** kombináció. A két MI-kódot
ugyanannak a tárgycsaládnak tekintem. Az első és az ismételt felvételek is szerepelnek.

A célérték meghatározása:
1. Az elismert MI-eredményeket nem kezelem újonnan megszerzett jegyként.
2. Elsőként az utolsó érvényes numerikus jegyet választom. Ha nincs ilyen,
   az ismeretlen érvényességű jegyek közül választok időrendben.
3. Ha csak érvénytelen jegyek maradnak, a célérték bizonytalan: a mintát kihagyom.
4. Jegy hiányában a megtagadott aláírás önálló eredmény, akkor is, ha az export
   annak érvényességét hamisnak jelöli. Az egyszerűen hiányzó jegy nem elégtelen.
5. Az időbélyeg nélküli, ellentmondó vagy a becslés előtt már ismert eredményeket
   külön számolom, és nem használom célként.

**Becslési időpontok:** őszi félévben szeptember 1., tavasziban február 1.
Ezek elemzési határnapok, nem az egyetem hivatalos félévkezdő dátumai.

In [ ]:
def folded(value):
    value = "" if pd.isna(value) else str(value)
    return " ".join("".join(c for c in unicodedata.normalize("NFD", value.casefold())
                            if unicodedata.category(c) != "Mn").split())

def term_number(term):
    match = re.fullmatch(r"(\d{4})/(\d{2})/([12])", str(term))
    if not match or int(match[2]) != (int(match[1]) + 1) % 100:
        return np.nan
    return int(match[1]) * 2 + int(match[3]) - 1

def prediction_time(term):
    number = term_number(term)
    if pd.isna(number):
        raise ValueError("Hibás félév a tanítómintában.")
    year, half = divmod(int(number), 2)
    return pd.Timestamp(year, 9, 1) if half == 0 else pd.Timestamp(year + 1, 2, 1)

GRADE_LABELS = {1: "Elégtelen (1)", 2: "Elégséges (2)", 3: "Közepes (3)", 4: "Jó (4)", 5: "Jeles (5)"}
SUBJECT_PATTERNS = {
    "programozas_ii": r"^programozas\s+ii(?:\b|[.(])",
    "algoritmusok_i": r"^adatstrukturak es algoritmusok\s+i(?:\b|[.(])",
    "diszkret_matematika": r"^diszkret matematika\b",
}

def subject_family(name):
    normalised = folded(name)
    return next((family for family, pattern in SUBJECT_PATTERNS.items()
                 if re.search(pattern, normalised)), "")

def prepare_records(frame):
    frame = frame.copy()
    frame["kepzes_kod"] = frame.kepzes_kod.fillna("ismeretlen").astype(str)
    frame["_result_time"] = pd.to_datetime(frame.eredmeny_datuma, errors="coerce")
    frame["_recorded_time"] = pd.to_datetime(frame.rogzites_datuma, errors="coerce")
    frame["_term"] = frame.felev.map(term_number)
    frame["_family"] = frame.targynev.map(subject_family)
    for column in ["jegy", "kredit", "ervenyes", "elismert"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame

def select_outcome(group, cutoff):
    grades = group[(group.eredmeny_statusz == "jegy") & (group.elismert != 1)]
    valid = grades[grades.ervenyes == 1]
    unknown_validity = grades[grades.ervenyes.isna()]
    if not valid.empty:
        pool, source = valid, "ervenyes_jegy"
    elif not unknown_validity.empty:
        pool, source = unknown_validity, "ismeretlen_ervenyesseg"
    elif not grades.empty:
        return None, "csak_ervenytelen_jegy"
    elif ((group.eredmeny_statusz == "jegy") & (group.elismert == 1)).any():
        return None, "csak_elismert_jegy"
    else:
        pool = group[group.eredmeny_statusz == "alairas_megtagadva"]
        source = "alairas_megtagadva"
    if pool.empty:
        return None, "nincs_ismert_eredmeny"
    if pool._result_time.isna().any():
        return None, "hianyzo_celdatum"
    ordered = pool.sort_values(["_result_time", "_recorded_time"], na_position="first")
    selected = ordered.iloc[-1]
    tied = ordered[ordered._result_time.eq(selected._result_time)
                   & ordered._recorded_time.fillna(pd.Timestamp.min).eq(
                       selected._recorded_time if pd.notna(selected._recorded_time) else pd.Timestamp.min)]
    if tied.jegy.nunique(dropna=False) > 1:
        return None, "ellentmondo_celertek"
    if selected._result_time < cutoff:
        return None, "a_becsleskor_mar_ismert_eredmeny"
    return selected, source

def history_features(history, cutoff, current_term, mi_codes):
    # Reconstruct only what was already recorded before this semester.
    known = history[(history._term < current_term)
                    & (history._result_time < cutoff)
                    & (history._recorded_time < cutoff)].copy()
    known = known.drop_duplicates(["targykod", "felev", "eredmeny_statusz", "jegy",
                                    "_result_time", "_recorded_time"])
    known = known.sort_values(["_result_time", "_recorded_time"])
    grades = known[known.eredmeny_statusz == "jegy"].drop_duplicates(
        ["targykod", "felev"], keep="last")
    signatures = known[known.eredmeny_statusz.isin(["alairva", "alairas_megtagadva"])]
    signatures = signatures.drop_duplicates(["targykod", "felev"], keep="last")
    observed_courses = len(known.drop_duplicates(["targykod", "felev"]))
    refused_courses = int((signatures.eredmeny_statusz == "alairas_megtagadva").sum())
    result = {
        "korabbi_atlag": grades.jegy.mean(),
        "korabbi_elegtelen_arany": (grades.jegy == 1).mean() if len(grades) else np.nan,
        "korabbi_megtagadas_arany": refused_courses / observed_courses if observed_courses else np.nan,
        # Diagnostic columns: never included in FEATURE_GROUPS.
        "korabbi_jegyek_szama": len(grades),
        "korabbi_mi_felvetelek": known.loc[known.targykod.isin(mi_codes), "felev"].nunique(),
    }
    for family in SUBJECT_PATTERNS:
        subject = grades[grades._family == family]
        result[f"targy_{family}"] = subject.jegy.iloc[-1] if len(subject) else np.nan
    result["_last_history_result"] = known._result_time.max()
    result["_last_history_recorded"] = known._recorded_time.max()
    return result

def build_samples(frame, mi_codes):
    frame = prepare_records(frame)
    all_history = {sid: group for sid, group in frame.groupby("hallgato_id", sort=False)}
    mi = frame[frame.targykod.isin(mi_codes)]
    output, excluded = [], Counter()
    for (sid, programme, term), group in mi.groupby(["hallgato_id", "kepzes_kod", "felev"], sort=False):
        cutoff = prediction_time(term)
        selected, reason = select_outcome(group, cutoff)
        if selected is None:
            excluded[reason] += 1
            continue
        row = {
            "hallgato_id": sid, "kepzes_kod": programme, "felev": term,
            "tanev": str(term)[:7], "tanev_kezdete": int(str(term)[:4]),
            "prediction_date": cutoff, "target_source": reason,
            "target_known_at": max(t for t in [selected._result_time, selected._recorded_time] if pd.notna(t)),
            "eredmeny": "Megtagadva" if reason == "alairas_megtagadva" else GRADE_LABELS[int(selected.jegy)],
            "cel_jegybejegyzesek_szama": int((group.eredmeny_statusz == "jegy").sum()),
        }
        row.update(history_features(all_history[sid], cutoff, term_number(term), mi_codes))
        for column in ["matek_erettsegi_szint", "info_erettsegi_szint", "kollegium_elorejelzeskor"]:
            value = selected.get(column, np.nan)
            row[column] = "ismeretlen" if pd.isna(value) or str(value).strip() == "" else str(value)
        output.append(row)
    samples = pd.DataFrame(output).sort_values(["tanev_kezdete", "felev", "kepzes_kod", "hallgato_id"]).reset_index(drop=True)
    return samples, excluded

def split_by_academic_year(samples):
    years = sorted(samples.tanev_kezdete.unique())
    if len(years) < 3:
        raise ValueError("Legalább három tanév szükséges.")
    valid_year, test_year = years[-2:]
    splits = {"train": samples[samples.tanev_kezdete < valid_year].copy(),
              "valid": samples[samples.tanev_kezdete == valid_year].copy(),
              "test": samples[samples.tanev_kezdete == test_year].copy()}
    assert sum(map(len, splits.values())) == len(samples)
    return splits, int(valid_year), int(test_year)

In [ ]:
samples, excluded = build_samples(records, MI_CODES)
splits, VALID_YEAR, TEST_YEAR = split_by_academic_year(samples)
train, valid, test = (splits[name] for name in ["train", "valid", "test"])
assert VALID_YEAR == 2024 and TEST_YEAR == 2025, "Megváltoztak a tanévek; ellenőrizd a felosztást!"
assert samples.eredmeny.isin(CLASS_ORDER).all()
assert (samples._last_history_result.dropna() < samples.loc[samples._last_history_result.notna(), "prediction_date"]).all()
assert (samples._last_history_recorded.dropna() < samples.loc[samples._last_history_recorded.notna(), "prediction_date"]).all()
assert train.target_known_at.max() < pd.Timestamp(VALID_YEAR, 9, 1)
assert valid.target_known_at.max() < pd.Timestamp(TEST_YEAR, 9, 1)
display(pd.DataFrame([{"Halmaz": name, "Tanévek": ", ".join(sorted(part.tanev.unique())),
                       "Minták": len(part), "Hallgatók": part.hallgato_id.nunique()}
                      for name, part in splits.items()]))
display(pd.Series(excluded, name="Kihagyott MI-felvételek").to_frame())
display(pd.crosstab(train.eredmeny, columns="Train").reindex(CLASS_ORDER, fill_value=0))
# A training-only example of how several source entries produce one target.
example = train[train.cel_jegybejegyzesek_szama >= 2].iloc[0]
example_rows = records[(records.hallgato_id == example.hallgato_id)
                       & (records.kepzes_kod == example.kepzes_kod)
                       & (records.felev == example.felev) & records.targykod.isin(MI_CODES)]

## Jellemzők előállítása

A korábbi jegyekből csak olyan rekordot használok, amely **korábbi félévhez
tartozik, és az eredmény és a rögzítés dátuma is a becslési időpont előtti**.
Az akkori tudást rekonstruálom: a később megváltozhatott érvényességi jelzővel
nem szűröm a múltbeli bemeneteket. Az időbeli korlátot ellenőrző állítások is vizsgálják.

Három kis, egymásra épülő bemenetet hasonlítok össze:

- **A – Általános tanulmányi előzmények: 3 oszlop.** Átlag, elégtelenek aránya és megtagadott aláírások aránya.
- **B – Kapcsolódó tárgyak: 6 oszlop.** A + Programozás II., Adatstruktúrák és algoritmusok I., Diszkrét matematika.
- **C – Háttéradatok: 9 oszlop.** B + matematika- és informatikaérettségi szintje, valamint a becsléskori kollégium.

Az átlag és a súlyozott átlag, illetve a jegyszám, a kreditösszeg és a megfigyelt
félévek száma egymással részben átfedő információt adna. Az egyszerűbb változatban
ezeket nem halmozom. A megtagadásokat aránnyal mérem, hogy a hosszabb megfigyelési
időszak önmagában ne növelje az értéket. A tantárgyakat az MI-hez való tartalmi
kapcsolatuk alapján választottam. A képzéskód, a képzési szint, a tagozat és a
kollégiumi adatillesztés állapota nem modellbemenet. A kilenc változót tartalmi
megfontolás és a tanítóadatok vizsgálata alapján rögzítem; az A–B–C összevetést
a validáción végzem, nem a teszteredmény szerint válogatok oszlopokat.

### A bemeneti oszlopok jelentése

A darabszámok az **eredeti oszlopokra** vonatkoznak. A hiányjelzők és a kategóriák
0/1 kódolása miatt a feldolgozás utáni tanítómátrixnak több oszlopa lehet.

#### A – Általános tanulmányi előzmények

Az összesítésben **egy tantárgy egy félévben egyszer számít**. Ha valaki két
külön félévben is felvette ugyanazt a tárgyat, azt két külön tárgyfelvételként
kezelem. A jegyek összesítésekor egy tárgyfelvétel több jegye közül az utolsó
ismert jegyet használom.

| Oszlop neve | Adattípus | Tartalom |
|---|---|---|
| `korabbi_atlag` | Numerikus, folytonos (1–5) | A korábbi félévekben felvett tantárgyak utolsó ismert jegyeinek számtani átlaga. |
| `korabbi_elegtelen_arany` | Numerikus arány (0–1) | Az elégtelenek aránya a korábbi félévekben felvett tantárgyak utolsó ismert jegyei között. |
| `korabbi_megtagadas_arany` | Numerikus arány (0–1) | Azon korábbi tárgyfelvételek aránya az összes megfigyelt, időben felhasználható tárgyfelvétel között, amelyeknél az utolsó ismert aláírásbejegyzés megtagadás. |

A megtagadási arány számításakor minden olyan korábbi tárgyfelvétel egyszer
számít, amelyhez van időben felhasználható eredménybejegyzés. Több vizsgajegy vagy
ismételt aláírásbejegyzés nem növeli ezt a darabszámot; az aláírás későbbi, még
a határnap előtti javítása az utolsó aláírásállapoton keresztül számít.
Előzmény hiányában az arány hiányzó, nem nulla. A nevező az exportból
megfigyelhető előzményeket jelenti, nem egy teljes hivatalos tárgyfelvételi nyilvántartást.

#### B – Három kapcsolódó tárgy korábbi jegye

A B változat az A három oszlopát egészíti ki az alábbiakkal.

| Oszlop neve | Adattípus | Tartalom |
|---|---|---|
| `targy_programozas_ii` | Ordinális jegy, numerikusan tárolva (1–5) | A Programozás II. tárgycsoport időrendben utolsó ismert korábbi jegye. |
| `targy_algoritmusok_i` | Ordinális jegy, numerikusan tárolva (1–5) | Az Adatstruktúrák és algoritmusok I. tárgycsoport időrendben utolsó ismert korábbi jegye. |
| `targy_diszkret_matematika` | Ordinális jegy, numerikusan tárolva (1–5) | A Diszkrét matematika tárgycsoport időrendben utolsó ismert korábbi jegye. |

A név szerinti csoportok nem hivatalos tantervi ekvivalenciák. A hiányzó jegy
nem elégtelen: a tanítóadatból tanult mediánnal pótolom, és hiányjelzőt is készítek.
A numerikus kezelés az egymást követő jegyek közti különbséget azonos lépésnek tekinti.

#### C – Három kiegészítő háttéradat

A C változat a B hat oszlopa mellé kapja az alábbiakat.

| Oszlop neve | Adattípus | Tartalom |
|---|---|---|
| `matek_erettsegi_szint` | Kategorikus: közép / emelt / ismeretlen | A képzéshez és felvételhez illesztett matematikaérettségi szintje, két ismert szint esetén az emeltet megtartva. |
| `info_erettsegi_szint` | Kategorikus: közép / emelt / ismeretlen | A képzéshez és felvételhez illesztett informatika- vagy digitális kultúra érettségi szintje, két ismert szint esetén az emeltet megtartva. |
| `kollegium_elorejelzeskor` | Kategorikus, nominális | A becslési határnapra egyértelműen illesztett kollégium neve, megfelelő találat hiányában ismeretlen értékkel. |

Az ismeretlen érettségi nem jelent középszintet, az ismeretlen kollégium pedig
nem jelenti azt, hogy a hallgató nem volt kollégista. A jegyszerzéskori kollégium
helyett a becsléskor ismertet használom, hogy későbbi adat ne kerüljön a bemenetbe.

### Azonosítók, célváltozó és leíró oszlopok

Az alábbi oszlopokat mintaképzésre, időbeli ellenőrzésre vagy utólagos elemzésre
használom. **Egyik sem magyarázóváltozó**; az `eredmeny` az elkülönített célváltozó.

| Oszlop neve | Adattípus | Tartalom |
|---|---|---|
| `hallgato_id` | Szöveges, álnevesített kulcs | A hallgató bejegyzéseit összekapcsoló HMAC-azonosító. |
| `kepzes_kod` | Szöveges kulcs | Az MI-felvétel képzése, amely a minták egyértelmű megkülönböztetéséhez szükséges. |
| `felev` | Rendezett időbeli kategória | Az MI-felvétel féléve, például 2024/25/2. |
| `tanev` | Időbeli kategória | Az MI-felvétel tanéve, amely meghatározza a tanító-, validációs vagy teszthalmazt. |
| `tanev_kezdete` | Diszkrét évszám | A tanév kezdő éve az időrendi felosztáshoz és rendezéshez. |
| `prediction_date` | Dátum | A félév becslési határnapja, amelynél csak korábbi előzmények használhatók. |
| `target_source` | Kategorikus állapotjelző | A célérték kiválasztásának szabálya, például érvényes jegy vagy megtagadott aláírás. |
| `target_known_at` | Dátum | A választott célbejegyzés eredmény- és rögzítési dátuma közül a későbbi. |
| `eredmeny` | Kategorikus célváltozó, 6 osztály | Az öt MI-jegy valamelyike vagy az aláírás megtagadása. |
| `cel_jegybejegyzesek_szama` | Diszkrét darabszám | Az MI-célfélév numerikus jegybejegyzéseinek száma utólagos leíráshoz. |
| `korabbi_jegyek_szama` | Diszkrét darabszám | Az ismert jeggyel rendelkező korábbi tárgyfelvételek száma az előzményhiány ellenőrzéséhez. |
| `korabbi_mi_felvetelek` | Diszkrét darabszám | Az időben megfigyelt korábbi MI-félévek száma az ismétlések leírásához. |
| `_last_history_result` | Dátum | A felhasznált előzmények legutolsó eredménydátuma az időbeli ellenőrzéshez. |
| `_last_history_recorded` | Dátum | A felhasznált előzmények legutolsó rögzítési dátuma az időbeli ellenőrzéshez. |

A jegybejegyzések száma nem teljes vizsgamegjelenési napló. A megajánlott jegyet
a forrás nem jelöli megbízhatóan, ezért nem találok ki ilyen értéket.

In [ ]:
SUMMARY_FEATURES = ["korabbi_atlag", "korabbi_elegtelen_arany", "korabbi_megtagadas_arany"]
SUBJECT_FEATURES = [f"targy_{family}" for family in SUBJECT_PATTERNS]
CONTEXT_FEATURES = ["matek_erettsegi_szint", "info_erettsegi_szint", "kollegium_elorejelzeskor"]
FEATURE_GROUPS = {"A": SUMMARY_FEATURES, "B": SUMMARY_FEATURES + SUBJECT_FEATURES,
                  "C": SUMMARY_FEATURES + SUBJECT_FEATURES + CONTEXT_FEATURES}
assert [len(FEATURE_GROUPS[group]) for group in ["A", "B", "C"]] == [3, 6, 9]
metadata_columns = {"hallgato_id", "kepzes_kod", "eredmeny", "felev", "tanev", "tanev_kezdete",
                    "cel_jegybejegyzesek_szama", "korabbi_jegyek_szama", "korabbi_mi_felvetelek"}
assert all(not metadata_columns.intersection(columns) for columns in FEATURE_GROUPS.values())
for column in SUMMARY_FEATURES + SUBJECT_FEATURES:
    samples[column] = pd.to_numeric(samples[column], errors="coerce").astype(float)
    for part in splits.values():
        part[column] = pd.to_numeric(part[column], errors="coerce").astype(float)
for part in splits.values():
    for column in CONTEXT_FEATURES:
        part[column] = part[column].astype(object).where(part[column].notna(), "ismeretlen")
display(pd.DataFrame({"Változat": list(FEATURE_GROUPS),
                      "Bemeneti oszlopok": [len(v) for v in FEATURE_GROUPS.values()]}))
display(train[SUBJECT_FEATURES].isna().mean().mul(100).rename("Hiány a train halmazban (%)").round(1).to_frame())
display(train.groupby("eredmeny").agg(Minták=("eredmeny", "size"),
    Jegybejegyzések_átlaga=("cel_jegybejegyzesek_szama", "mean"),
    Jegybejegyzések_mediánja=("cel_jegybejegyzesek_szama", "median"))
    .reindex(CLASS_ORDER).round(2))
display(Markdown("A jegybejegyzések száma utólagos leírás, nem modellbemenet és nem teljes vizsgamegjelenési napló."))
fig, ax = plt.subplots(figsize=(9, 4))
train.eredmeny.value_counts().reindex(CLASS_ORDER, fill_value=0).plot.bar(ax=ax, color="#4477AA")
ax.set(xlabel="MI-eredmény", ylabel="Tanítóminták", title="A tanítóhalmaz osztályeloszlása")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(RESULT_DIR / "train_class_distribution.png", dpi=150)
plt.show()

## Tanító-, validációs és teszthalmaz

A **2017/18–2023/24** tanévek alkotják a tanítóhalmazt, **2024/25** a validáció,
**2025/26** a teszt. Mindkét félév ugyanabban a tanéves halmazban marad.

Egy ismétlő hallgató több halmazban is előfordulhat: ez a kért tanéves felosztás
és a következő év visszatérő hallgatóinak előrejelzése miatt szándékos.
A hallgató azonosítója nem bemenet. A teszt végén az új és visszatérő hallgatók
eredményét külön is közlöm; ez nem kizárólag új személyekre vonatkozó mérés.

A próbálgatáshoz mindig a validációt használom. A paraméterkeresés belső részfelosztásai
is időrendiek, és kizárólag a tanítóhalmaz tanéveiből készülnek. A végén a rögzített
konfigurációt a tanító- és validációs adatok egyesítésén újratanítom, és egyszer értékelem a teszten.
A teszttanév tavaszi mintáinak előzményeiben az addig már ismert őszi eredmények
is megjelenhetnek: ez félévenként frissített bemenet, nem újratanítás a teszten.

## Közös kiértékelő függvény

Minden modell ugyanazokat a mintákat és ugyanazt a kiértékelést kapja.
A numerikus bemeneteket a tanítóhalmaz mediánjával pótolom, és szükség esetén
standardizálom. A kategóriák külön 0/1 értékű jelzőoszlopokká alakulnak. Minden előfeldolgozás
a modell saját feldolgozási folyamatában tanulódik meg.

A fő mutató a **makro-F1**: mind a hat osztály azonos súlyt kap. Az összesített pontosság,
létszámmal súlyozott F1 és osztályonkénti felismerési arány segít értelmezni a különbséget.
A megtagadás külön kategória, nem nullás jegy; nem számolok rá jegyeltérést.

In [ ]:
scorer = make_scorer(f1_score, labels=CLASS_ORDER, average="macro", zero_division=0)
runs = []

def make_pipeline(estimator, feature_group="B", scale=True):
    columns = FEATURE_GROUPS[feature_group]
    numeric = [c for c in columns if c not in CONTEXT_FEATURES]
    categorical = [c for c in columns if c in CONTEXT_FEATURES]
    num_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), numeric)]
    if categorical:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="ismeretlen", keep_empty_features=True)),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=5)),
        ]), categorical))
    return Pipeline([("prep", ColumnTransformer(transformers, verbose_feature_names_out=False)),
                     ("model", clone(estimator))])

def metric_row(actual, predicted):
    return {"Accuracy": accuracy_score(actual, predicted),
            "Macro-F1": f1_score(actual, predicted, labels=CLASS_ORDER, average="macro", zero_division=0),
            "Weighted-F1": f1_score(actual, predicted, labels=CLASS_ORDER, average="weighted", zero_division=0),
            "Macro-recall": recall_score(actual, predicted, labels=CLASS_ORDER, average="macro", zero_division=0)}

def evaluate_model(name, estimator=None, group="B", scale=True, pipeline=None, experiment="baseline"):
    model = clone(pipeline) if pipeline is not None else make_pipeline(estimator, group, scale)
    columns = FEATURE_GROUPS[group]
    started = time.perf_counter()
    model.fit(train[columns], train.eredmeny)
    seconds = time.perf_counter() - started
    row = {"Modell": name, "Bemenet": group, "Kísérlet": experiment,
           **metric_row(valid.eredmeny, model.predict(valid[columns])),
           "Train Macro-F1": metric_row(train.eredmeny, model.predict(train[columns]))["Macro-F1"],
           "Tanítás (s)": seconds}
    runs.append({"row": row, "model": model, "group": group})
    return row

def show_results(rows):
    table = pd.DataFrame(rows).sort_values(["Macro-F1", "Accuracy"], ascending=False)
    display(table.reset_index(drop=True).round(3))
    return table

## Kiinduló modellek

Először rögzített, egyszerű beállításokkal hasonlítom össze az órán tanult
modelleket. A többségi alapmodell megmutatja, mit ér el egy olyan megoldás,
amely minden hallgatóra ugyanazt az eredményt jósolja.
A naiv Bayes-osztályozó különösen érdekes, mert a tantárgyjegyek egymással összefügghetnek,
miközben a modell feltételes függetlenséget feltételez.

In [ ]:
estimators = {
    "Többségi alapmodell": DummyClassifier(strategy="most_frequent"),
    "Döntési fa": DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=RANDOM_STATE),
    "Gaussian Naive Bayes": GaussianNB(),
    "kNN": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Logisztikus regresszió": LogisticRegression(C=1, max_iter=3000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8,
        min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=N_JOBS),
}
baseline_rows = [evaluate_model(name, estimator) for name, estimator in estimators.items()]
baseline_table = show_results(baseline_rows)

### Az eredménytábla mutatóinak jelentése

A táblázat pontossági mutatói a **validációs halmazra** vonatkoznak, kivéve a külön jelölt tanítási mutatót.
Az F1 a **találati pontosság** (az adott osztályba jósolt esetek közül a helyesek aránya) és a **felismerési arány** (az adott osztály valódi esetei közül a megtaláltak aránya) harmonikus átlaga.

| Mutató a táblázatban | Mit mutat meg? |
|---|---|
| `Accuracy` | Az összes minta hányadán egyezik a becsült és a tényleges osztály. |
| `Macro-F1` | Az osztályonkénti F1-értékek egyszerű átlaga, ezért mind a hat osztály, a ritkábbak is, azonos súlyt kapnak. |
| `Weighted-F1` | Az osztályonkénti F1-értékek mintaszámmal súlyozott átlaga, ezért a gyakoribb osztályok erősebben alakítják. |
| `Macro-recall` | Az osztályonkénti felismerési arányok egyszerű átlaga, vagyis egyenlő osztálysúllyal méri a valódi esetek megtalálását. |
| `Train Macro-F1` | A tanítóhalmazon számolt makro-F1, amelynek a validációs értékhez képesti nagy többlete túlillesztésre utalhat. |
| `Tanítás (s)` | A teljes előfeldolgozó és osztályozó folyamat illesztésének futásideje másodpercben. |

A pontossági mutatók 0 és 1 közöttiek, a nagyobb érték a jobb; a tanítási időnél a kisebb érték jelenti a gyorsabb futást.

### Az első eredmények

A többségi alapmodell validációs pontossága **29.4%**, makro-F1 értéke
viszont csak **0.076**. Egyetlen osztály jóslásával a többi eredmény
felismerése teljesen elmarad.

A hat bemeneti oszlopot használó induló modellek közül a **véletlen erdő** adta
a legjobb validációs makro-F1-et (**0.377**), **41.7%** pontossággal.
Ez kiinduló mérés: más bemenettel vagy beállítással változhat a sorrend.

A Gauss-féle naiv Bayes-osztályozó makro-F1-e **0.110**.
A jegyek diszkrét eloszlása, az imputálás és a bemenetek összefüggése is
eltérhet a modell feltételezéseitől. Ezek lehetséges magyarázatok; a mérés
önmagában nem bizonyítja, melyik okozta a gyenge eredményt.

## Famélység és túlillesztés

Ugyanazokkal a bemenetekkel csak a famélységet változtatom.
A tanítási és validációs makro-F1 különbsége segít észrevenni a túlillesztést.
A mélyebb fa eredményét itt még nem tekintem előre biztosan jobbnak vagy rosszabbnak.

In [ ]:
depth_rows = []
for depth in [2, 4, 8, None]:
    row = evaluate_model(f"Döntési fa, mélység={depth}", DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=1, random_state=RANDOM_STATE), experiment="tree_depth")
    depth_rows.append({**row, "Mélység": str(depth)})
show_results(depth_rows)
depth_table = pd.DataFrame(depth_rows)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(depth_table["Mélység"], depth_table["Train Macro-F1"], marker="o", label="Train")
ax.plot(depth_table["Mélység"], depth_table["Macro-F1"], marker="o", label="Validáció")
ax.set(xlabel="Maximális famélység", ylabel="Macro-F1", title="A famélység hatása")
ax.legend()
plt.tight_layout()
fig.savefig(RESULT_DIR / "tree_depth.png", dpi=150)
plt.show()

### Mit tanult meg a mély fa?

A korlátozatlan fa tanítási makro-F1-e **0.920**,
miközben a validációs értéke **0.297**. A tanítóadatok sokkal jobb
leírása nem jelent hasonló teljesítményt a következő tanéven.

A legjobb validációs értéket a **8** mélység adta
(**0.310**); ez csak kicsivel jobb a 4 mélységű fa eredményénél.
A korlátozatlan fa tanítási előnye már nem térült meg a validáción.
Egyetlen tanév kis különbségéből nem állítom, hogy a 8-as mélység általában optimális.

**A kNN a hasonló előzményű minták eredménye alapján becsül.** A neve *k legközelebbi szomszéd*: egy új hallgató bemeneti adataihoz megkeresi a tanítóhalmazban a `k` legközelebbi mintát, és azok eredményei szavaznak a becsült osztályra.

Például három szomszéd eredménye legyen kettes, kettes és négyes:

- Egyenlő szavazati súlynál a kettes nyer.
- Távolsággal súlyozva a közelebbi szomszéd erősebben szavaz. Ha a négyes nagyon közel van, akár az is nyerhet.

## A skálázás hatása

A jegyek és az átlag 1–5 közöttiek, az elégtelenek és a megtagadások aránya
viszont 0–1 közötti. A hiányjelzők is beleszólnak a távolságba.
Standardizálással megváltozik, mely eltérések kapnak nagyobb súlyt a kNN-ben.
Azonos szomszédszám és súlyozás mellett hasonlítom össze a nyers és a
standardizált B bemenetet; a skálázás javító hatását nem feltételezem előre.

In [ ]:
knn_rows = []
for scale in [False, True]:
    for k in [3, 7, 15, 31]:
        row = evaluate_model(f"kNN, k={k}, skálázás={scale}",
            KNeighborsClassifier(n_neighbors=k, weights="distance"), scale=scale, experiment="knn_scaling")
        knn_rows.append({**row, "k": k, "Skálázás": scale})
show_results(knn_rows)

### A távolság megváltoztatja a szomszédokat

Skálázás nélkül a legjobb kipróbált beállítás **k=31**, makro-F1:
**0.291**. Standardizálva a legjobb **k=7**, makro-F1:
**0.348**. A táblázat azonos k mellett is megmutatja a skálázás hatását.

A skálázás megváltoztatja az átlag, az arányok, a tárgyjegyek és a hiányjelzők
egymáshoz viszonyított súlyát. A javulás mértéke függ a szomszédszámtól;
a legjobb k-t és az előfeldolgozást ezért együtt érdemes vizsgálni.
A távolsággal súlyozott kNN magas tanítási mutatóját óvatosan kell olvasni:
tanítási adaton a minta saját maga is nulla távolságú szomszéd. A döntő mérés a validáció.

## További információk bevonása

Most nem az algoritmust változtatom, hanem azt, hogy mit tud a hallgatóról.
A logisztikus regresszió és a véletlen erdő ugyanazzal a beállítással kapja meg
az A, B és C bemenetet. Így látszik, hogy ezen a validáción mennyit tesznek hozzá
a konkrét tárgyjegyek, illetve a kiegészítő adatok.

In [ ]:
feature_rows = []
for name in ["Logisztikus regresszió", "Random Forest"]:
    for group in FEATURE_GROUPS:
        feature_rows.append(evaluate_model(name + " – " + group, estimators[name],
            group=group, experiment="feature_groups"))
feature_table = show_results(feature_rows)

### Több bemenetből lett-e jobb becslés?

A logisztikus regresszió validációs makro-F1 értékei:
**A: 0.314**,
**B: 0.261**,
**C: 0.313**.
A három általános jellemzővel kapott eredmény itt lényegében megegyezik a
kilencváltozós változatéval; a tárgyjegyek önmagukban történő hozzáadása rontott.

A véletlen erdő megfelelő értékei:
**A: 0.293**,
**B: 0.377**,
**C: 0.326**.
Ennél a modellnél a három kapcsolódó tárgy érdemi javulást hozott, a háttéradatok
hozzáadása viszont csökkentette a validációs makro-F1-et. A további kísérletekhez
ezért a **B, azaz hatoszlopos bemenetet** választottam.

A jellemzők haszna tehát modellfüggő: az erdő például eltérő értéktartományokban
más szabályokat tanulhat. A C gyengébb eredménye nem bizonyítja az érettségi vagy
a kollégium általános jelentéktelenségét. A hiányok, a kategóriák gyakorisága,
a tantervi eltérések és az adott validációs tanév is befolyásolhatják az eredményt.

## Class_weight: ritkább osztályok nagyobb súlyal

A ritkább eredmények felismerése az összesített pontosságban könnyen háttérbe szorulhat.
A `class_weight="balanced"` a ritkább osztályok hibáit nagyobb súllyal veszi
figyelembe. Az osztálysúlyozást a legjobb validációs bemeneti változaton próbálom
ki; nem feltételezem, hogy minden mutatót javít.

In [ ]:
weight_rows = []
best_group_so_far = feature_table.iloc[0]["Bemenet"]
for name in ["Logisztikus regresszió", "Random Forest"]:
    for weight in [None, "balanced"]:
        estimator = clone(estimators[name]).set_params(class_weight=weight)
        weight_rows.append(evaluate_model(f"{name}, class_weight={weight}", estimator,
            group=best_group_so_far, experiment="class_weight"))
show_results(weight_rows)

### Az osztálysúlyozás eredménye modellfüggő

A hatoszlopos B bemeneten a logisztikus regresszió makro-F1-e súlyozás nélkül
**0.261**, súlyozással **0.337** lett.
Közben a pontosság **38.6%**-ról **30.3%**-ra változott.
A ritka osztályok egyenlőbb figyelembevétele itt javította a fő mutatót,
miközben összességében kevesebb becslés lett helyes.

A véletlen erdőnél a makro-F1 **0.377**-ról **0.326**-ra,
a pontosság **41.7%**-ról **29.8%**-ra csökkent.
Az osztálysúlyozás nem garantál javulást; ezen a validáción az erdőnél a
súlyozás nélküli változat maradt erősebb. Ezért fő szempont a korábban rögzített
makro-F1, és mellette közlöm az összesített pontosságot is.

## Hiperparaméterek hangolása


- **kNN:** a négy szomszédszám és két súlyozás — egyenlő vagy távolság szerinti — mind a **8 kombinációját** kipróbáljuk.
- **Véletlen erdő:** a fák számát, mélységét, minimális levéllétszámát és osztálysúlyozását változtatjuk. A megadott **36 kombinációból 12-t** próbálunk ki véletlenszerűen, rögzített véletlenmaggal.

A keresés a kNN-nél **15 szomszédot és egyenlő súlyozást**, az erdőnél **150 fát, 8-as mélységet, legalább 5 mintás leveleket és osztálysúlyozás nélküli működést** választott.

In [ ]:
cv_years = sorted(train.tanev_kezdete.unique())[-3:]
cv_splits = []
for year in cv_years:
    fit_idx = np.flatnonzero(train.tanev_kezdete.to_numpy() < year)
    eval_idx = np.flatnonzero(train.tanev_kezdete.to_numpy() == year)
    assert len(fit_idx) and len(eval_idx)
    assert train.iloc[fit_idx].tanev_kezdete.max() < train.iloc[eval_idx].tanev_kezdete.min()
    assert train.iloc[fit_idx].target_known_at.max() < pd.Timestamp(int(year), 9, 1)
    cv_splits.append((fit_idx, eval_idx))
display(pd.DataFrame([{"Fold": i + 1, "Tanító tanévek": ", ".join(sorted(train.iloc[a].tanev.unique())),
                       "Értékelési tanév": train.iloc[b].tanev.iloc[0], "Train n": len(a), "Eval n": len(b)}
                      for i, (a, b) in enumerate(cv_splits)]))
parameter_spaces = {
    "Döntési fa": {"model__max_depth": [2, 4, 8, None], "model__min_samples_leaf": [1, 5, 15]},
    "Gaussian Naive Bayes": {"model__var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5]},
    "kNN": {"model__n_neighbors": [3, 7, 15, 31], "model__weights": ["uniform", "distance"]},
    "Logisztikus regresszió": {"model__C": [0.1, 1, 10], "model__class_weight": [None, "balanced"]},
    "Random Forest": {"model__n_estimators": [150, 300], "model__max_depth": [4, 8, None],
                      "model__min_samples_leaf": [2, 5, 10], "model__class_weight": [None, "balanced"]},
}
tuning_names = baseline_table[baseline_table.Modell != "Többségi alapmodell"].head(2).Modell.tolist()
tuning_rows, tuning_details = [], []
for name in tuning_names:
    estimator = clone(estimators[name])
    if name == "Random Forest":
        estimator.set_params(n_jobs=1)
    pipe = make_pipeline(estimator, best_group_so_far)
    search_kwargs = dict(estimator=pipe, scoring=scorer, cv=cv_splits, n_jobs=N_JOBS,
                         refit=True, return_train_score=True, error_score="raise")
    if name == "Random Forest":
        search = RandomizedSearchCV(param_distributions=parameter_spaces[name], n_iter=12,
                                     random_state=RANDOM_STATE, **search_kwargs)
    else:
        search = GridSearchCV(param_grid=parameter_spaces[name], **search_kwargs)
    # Threads avoid Windows/Python 3.14 process-pool cleanup issues on this small dataset.
    with joblib.parallel_backend("threading", n_jobs=N_JOBS):
        search.fit(train[FEATURE_GROUPS[best_group_so_far]], train.eredmeny)
    row = evaluate_model(name + " – hangolt", pipeline=search.best_estimator_,
                         group=best_group_so_far, experiment="hyperparameters")
    tuning_rows.append(row)
    tuning_details.append({"model": name, "best_cv_macro_f1": float(search.best_score_),
                           "parameters": search.best_params_})
show_results(tuning_rows)
display(pd.DataFrame(tuning_details))

## A jellemzők fontossága


**A jellemzőfontosságnál azt mérjük, mennyire támaszkodik a kiválasztott modell egy oszlop információjára.** Ehhez *permutációs fontosságot* használunk:

1. Megmérjük a betanított modell validációs makro-F1-ét.
2. Egyetlen bemeneti oszlop értékeit véletlenszerűen összekeverjük a minták között. Például valaki más korábbi átlaga kerül az adott hallgatóhoz.
3. A többi oszlopot és a tényleges eredményeket megtartjuk, majd ugyanazzal a modellel újra becslünk.
4. Megnézzük, mennyivel csökkent a makro-F1.
5. Ezt oszloponként **10 alkalommal** megismételjük.

In [ ]:
comparison = pd.DataFrame([run["row"] for run in runs]).sort_values(
    ["Macro-F1", "Accuracy", "Modell"], ascending=[False, False, True]).reset_index(drop=True)
display(comparison.head(12).round(3))
comparison.to_csv(RESULT_DIR / "validation_comparison.csv", index=False, encoding="utf-8-sig")
winner_row = comparison.iloc[0]
winner = next(run for run in runs if run["row"]["Modell"] == winner_row.Modell
              and run["group"] == winner_row.Bemenet
              and run["row"]["Kísérlet"] == winner_row["Kísérlet"])
selected_features = FEATURE_GROUPS[winner["group"]]
selected_pipeline = clone(winner["model"])
with joblib.parallel_backend("threading", n_jobs=N_JOBS):
    perm = permutation_importance(winner["model"], valid[selected_features], valid.eredmeny,
                                  scoring=scorer, n_repeats=10, random_state=RANDOM_STATE, n_jobs=N_JOBS)
importance = pd.DataFrame({"Változó": selected_features, "Macro-F1 csökkenése": perm.importances_mean,
                           "Szórás": perm.importances_std}).sort_values("Macro-F1 csökkenése", ascending=False)
display(importance.head(12).round(4))
importance.to_csv(RESULT_DIR / "permutation_importance.csv", index=False, encoding="utf-8-sig")
plot_data = importance.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_data["Változó"], plot_data["Macro-F1 csökkenése"], xerr=plot_data["Szórás"], color="#4477AA", capsize=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(xlabel="Validációs macro-F1 átlagos csökkenése", title="Permutation importance")
plt.tight_layout()
fig.savefig(RESULT_DIR / "permutation_importance.png", dpi=150)
plt.show()

### A kiválasztott konfiguráció

A külső validáció alapján a választás a **hangolt véletlen erdő**, a **B**
bemenet **hat oszlopával**. Validációs makro-F1: **0.382**,
pontosság: **42.1%**.
A belső időrendi keresés **150 fát**,
**8 maximális mélységet**, **5 minimális levéllétszámot**
és osztálysúlyozás nélküli beállítást választott.
Az induló erdő **0.377** makro-F1-éhez képest a hangolás előnye kicsi;
ez nem bizonyít tartós fölényt minden következő tanéven.

A legnagyobb átlagos permutációs fontosságok:
**`korabbi_atlag` (0.150)**,
**`korabbi_megtagadas_arany` (0.113)** és
**`korabbi_elegtelen_arany` (0.073)**.
Ezek megkeverése rontotta leginkább a betanított modell validációs makro-F1-ét;
a három tárgyjegy megkeverése is pozitív átlagos csökkenést adott.
A fontosság a már kiválasztott modellre vonatkozik, nem mondja meg az összefüggés
irányát, és nem bizonyít okozást.

## Végső kiértékelés

A konfigurációt ezen a ponton rögzítettem. A kiválasztott feldolgozási folyamat friss
példányát a tanító- és a validációs halmaz egyesítésén tanítom újra; az imputálás és a kódolás
is ezen az egyesített, a tesztet megelőző időszakon illeszkedik újra.
Ezután kap először tesztbecslést a 2025/26-os tanév.
A korábban rögzített többségi alapmodell ugyanilyen időbeli tanítás után ad
viszonyítási alapot; a teszt alapján nem választok másik konfigurációt.

A teszt eredménye alapján már nem módosítok a modellen. Az alcsoportok és a
tévesztések alábbi vizsgálata ennek az egyetlen, rögzített becslésnek a leírása.

In [ ]:
development = pd.concat([train, valid], ignore_index=True)
final_model = clone(selected_pipeline)
final_model.fit(development[selected_features], development.eredmeny)
test_predictions = final_model.predict(test[selected_features])
final_metrics = metric_row(test.eredmeny, test_predictions)
final_baseline = make_pipeline(DummyClassifier(strategy="most_frequent"), winner["group"])
final_baseline.fit(development[selected_features], development.eredmeny)
test_baseline_metrics = metric_row(test.eredmeny, final_baseline.predict(test[selected_features]))
display(Markdown('### 1. táblázat – Összesített teszteredmények\n\nA többségi alapmodell és a kiválasztott modell eredménye ugyanazon a 2025/26-os teszthalmazon. A `Bemenet` az alkalmazott jellemzőcsoportot jelöli; az `Accuracy` a helyes becslések aránya, a `Macro-F1` az osztályok egyenlő súlyú F1-átlaga, a `Weighted-F1` a létszámmal súlyozott F1, a `Macro-recall` pedig az osztályonkénti felismerési arányok átlaga.'))
display(pd.DataFrame([
    {"Modell": "Többségi alapmodell", "Bemenet": winner["group"], "Teszttanév": test.tanev.iloc[0], **test_baseline_metrics},
    {"Modell": winner_row.Modell, "Bemenet": winner["group"], "Teszttanév": test.tanev.iloc[0], **final_metrics},
]).round(3))
report = classification_report(test.eredmeny, test_predictions, labels=CLASS_ORDER,
                               output_dict=True, zero_division=0)
class_report = pd.DataFrame(report).T.loc[CLASS_ORDER, ["precision", "recall", "f1-score", "support"]]
display(Markdown('### 2. táblázat – Osztályonkénti teszteredmények\n\nMinden sor egy tényleges MI-eredményosztályt ír le. A `precision` megmutatja, hogy az adott osztályba jósolt esetek mekkora része helyes; a `recall`, hogy az osztály valódi eseteinek mekkora részét találta meg a modell; az `f1-score` ezek harmonikus átlaga. A `support` az adott osztály tényleges mintaszáma a teszten, nem a jóslatok száma.'))
display(class_report.round(3))
display(Markdown('### Ábra – Tévesztési mátrix a teszthalmazon\n\nA sorok a tényleges, az oszlopok a becsült eredményt mutatják; a cellákban esetszámok szerepelnek. A bal alsó saroktól a jobb felső sarokig tartó átló helyes becsléseket, az azon kívüli cellák tévesztéseket tartalmaznak. Például az 1-es sor és 2-es oszlop metszete azt számolja, hány elégtelent becsült elégségesnek a modell. A megtagadott aláírás külön osztály.'))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(test.eredmeny, test_predictions, labels=CLASS_ORDER,
                                        display_labels=["Megtagadva", "1", "2", "3", "4", "5"],
                                        cmap="Blues", colorbar=False, ax=ax)
ax.set(title="Végső teszt – 2025/26", xlabel="Becsült eredmény", ylabel="Tényleges eredmény")
ax.grid(False)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(RESULT_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()

prediction_table = test[["hallgato_id", "kepzes_kod", "felev", "eredmeny"]].copy()
prediction_table["becsult_eredmeny"] = test_predictions
seen_ids = set(development.hallgato_id)
prediction_table["korabbi_tanitomintaban_szerepelt"] = test.hallgato_id.isin(seen_ids).to_numpy()
subgroup_rows = []
for seen, group in prediction_table.groupby("korabbi_tanitomintaban_szerepelt"):
    subgroup_rows.append({"Csoport": "Korábban is szereplő" if seen else "Új hallgató",
                          "Minták": len(group), **metric_row(group.eredmeny, group.becsult_eredmeny)})
display(Markdown('### 3. táblázat – Új és korábban is szereplő hallgatók\n\nUgyanannak a rögzített tesztbecslésnek a mutatói két csoportra bontva, a csoportonkénti mintaszámmal együtt. A „Korábban is szereplő” csoport azokat jelenti, akiknek azonosítója előfordult a tanító- és validációs MI-minták között. Az „Új hallgató” ezekben a mintákban még nem szerepelt; ettől más tárgyakból már lehetett ismert előzménye. Ez nem az első egyetemi félév vagy az első MI-felvétel szerinti bontás.'))
display(pd.DataFrame(subgroup_rows).round(3))
prediction_table.to_csv(RESULT_DIR / "test_predictions.csv", index=False, encoding="utf-8-sig")
wrong = test.loc[test.eredmeny.to_numpy() != test_predictions,
                 ["felev", "eredmeny", "korabbi_atlag", "korabbi_mi_felvetelek"]].copy()
wrong["Becsült"] = test_predictions[test.eredmeny.to_numpy() != test_predictions]
display(Markdown('### 4. táblázat – Példák a hibás becslésekre\n\nAz első nyolc téves tesztbecslés részletei: a félév (`felev`), a tényleges eredmény (`eredmeny`), a korábbi tanulmányi átlag (`korabbi_atlag`), a korábban megfigyelt MI-félévek száma (`korabbi_mi_felvetelek`) és a becsült eredmény (`Becsült`). Az MI-félévek száma leíró adat, nem modellbemenet és nem vizsgapróbálkozás-szám. Ezek kiválasztott példák, nem a teljes hibahalmaz összesítése.'))
display(wrong.head(8).reset_index(drop=True).round(2))

### Mit tudott a következő tanéven?

A 2025/26-os teszten a pontosság **34.6%**, a makro-F1 **0.341**.
A tanítási időszak alapján rögzített többségi alapmodell megfelelő értékei
**22.9%** és **0.062**.
A modell mindkét mutatóban jobb az alapmodellnél, de a hatosztályos becslés
pontossága korlátozott. A validációs makro-F1-hez képesti csökkenés a következő
tanévre való általánosítás korlátját mutatja.

**A megtagadás felismerése:** felismerési arány = **22.7%**,
találati pontosság = **71.4%**.
Sok megtagadást nem talál meg a modell. Az elégtelenek felismerési aránya is
alacsony (**8.0%**).

**Az elégségesek:** felismerési arány = **79.6%**, találati
pontosság = **27.3%**. A modell sok kettest jósol,
és közben más osztályok hallgatóit is ebbe a kategóriába sorolja.
**A jelesek** felismerési aránya **54.5%**. Az osztályonkénti mutatókat
mindig a létszámmal és a tévesztési mátrixszal együtt értelmezem.

Az új és visszatérő hallgatók külön táblázata ugyanennek a rögzített becslésnek
a leírása. Az eltérő osztályarányok és kis alcsoportlétszámok miatt a két csoport
egy-egy összesített mutatójából nem következtetek általános fölényre.

## Következtetések és továbbfejlesztési lehetőségek

A **3–6–9 oszlopos** kísérletek közül a kiválasztott konfiguráció a
**hatváltozós B bemenetet használó, hangolt véletlen erdő** lett.
Teszt-makro-F1 értéke **0.341**, pontossága **34.6%**.
A korábbi tanulmányi eredmények hordoznak prediktív információt, de az öt jegy
és a megtagadás pontos elkülönítése ezen az adaton nehéz maradt.

A kísérletek fő tanulságai:

- A korlátozatlan döntési fa tanítási előnye nem jelentett jobb validációs eredményt.
- A kNN-nél a skálázás és a szomszédszám együtt alakítja az eredményt.
- Az erdőnél a három kapcsolódó tárgyjegy javított, a három háttéradat már nem; a logisztikus regresszió másképp reagált ugyanazokra a bemenetekre.
- Az osztálysúlyozás javította a logisztikus regresszió makro-F1-ét, miközben csökkent a pontossága; az erdő mindkét mutatója romlott.
- A hangolás kis validációs javulást adott. A kiválasztott erdőben az átlag, a megtagadási arány és az elégtelenek aránya adta a legnagyobb permutációs fontosságot.
- A teszten az elégtelenek és a megtagadások felismerése gyenge maradt; az összesített mutatók mellett ezért szükséges az osztályonkénti elemzés.

**Továbbfejlesztési lehetőségek:** ellenőrzött tantervi tárgymegfeleltetések;
további tanévek; tanulási görbék és bizonytalansági intervallumok; külön vizsgálat
az első MI-felvételekre. A jegyek szórása külön, egyetlen változó hozzáadásával
végzett validációs kísérlet lehetne: a mostani egyszerűsítés nem bizonyítja,
hogy ez az információ minden modellnél felesleges. Ezek további kísérletek,
nem a mostani teszteredményhez igazított módosítások.

A következtetések a rögzített adatlenyomathoz és beállításokhoz tartoznak.
Adat- vagy paramétermódosítás után a szöveges értelmezést is frissíteni kell.

In [ ]:
# Save reproducible, pseudonymised artefacts. The secret key is not part of these files.
for name, part in splits.items():
    exported = part[["hallgato_id", "kepzes_kod", "felev", "tanev", "eredmeny"] + FEATURE_GROUPS["C"]]
    exported.to_csv(RESULT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
metadata = {
    "data_sha256": DATA_SHA256, "versions": versions, "random_state": RANDOM_STATE,
    "split_counts": {name: len(part) for name, part in splits.items()},
    "split_years": {name: sorted(part.tanev.unique().tolist()) for name, part in splits.items()},
    "excluded_targets": dict(excluded), "class_order": CLASS_ORDER,
    "selected_configuration": winner_row.Modell, "selected_features": selected_features,
    "feature_group": winner["group"], "feature_groups": FEATURE_GROUPS, "test_metrics": final_metrics, "test_baseline_metrics": test_baseline_metrics,
    "new_returning_test": subgroup_rows, "tuning": tuning_details,
    "baseline": baseline_rows, "tree_depth": depth_rows, "knn": knn_rows,
    "feature_experiments": feature_rows, "weight_experiments": weight_rows,
    "validation_winner": winner["row"], "top_importance": importance.head(8).to_dict("records"),
    "test_class_report": class_report.to_dict("index"),
    "no_history": {name: int((part.korabbi_jegyek_szama == 0).sum()) for name, part in splits.items()},
}
with (RESULT_DIR / "summary.json").open("w", encoding="utf-8") as stream:
    json.dump(metadata, stream, ensure_ascii=False, indent=2, default=str)
# Store the estimator, preprocessing and selected columns together.
# A fitted pipeline's classes are standard sklearn estimators; no notebook helper is pickled.
joblib.dump({"pipeline": final_model, "features": selected_features,
             "class_order": CLASS_ORDER, "data_sha256": DATA_SHA256}, RESULT_DIR / "final_model.joblib")
print("A train, valid, test halmazok és az eredmények helye: data/notebook_results/")
print("Notebook ellenőrzések: időrendi felosztás, korábbi előzmények, azonosító nélküli bemenetek – OK.")